In [ ]:
# Loading the dataset in - since it is so large, it must be done using batch processing

import pandas as pd
import numpy as np
import kagglehub


csv_path = kagglehub.dataset_download(
    "sobhanmoosavi/us-accidents",
    path="US_Accidents_March23.csv",
)

batch_size = 5000
batches = []
batch_num = 0

# To reduce the memory usage, we will do the filtering of the cities when processing
# We chose these cities because they are the cities with most data in dataset, and also are distinct and large population centers
cities = ["Miami", "Houston"]
filtered_batches = []

for batch in pd.read_csv(csv_path, chunksize=50000):
    filtered = batch[batch["City"].isin(cities)]
    
    if not filtered.empty:
        filtered_batches.append(filtered)

df_trimmed = pd.concat(filtered_batches, ignore_index=True)

print(df_trimmed.shape)

In [ ]:
# select columns to be dropped
columns_dropped = [
    # zero/near-zero variance
    'Turning_Loop',
    'Roundabout',
    'No_Exit',
    'Traffic_Calming',
    'Bump',

    # redundant or irrelevant
    'Country',
    'State',
    'End_Lat',
    'End_Lng',
    'Airport_Code',
    'Weather_Timestamp',
    'County',
    'Timezone',
    'Civil_Twilight',
    'Nautical_Twilight',
    'Astronomical_Twilight',
    'Wind_Chill(F)',
    'ID',
    'Description',
    'Street',
]

# # create a copy to keep a primary df and apply transformation
d2 = df_trimmed.copy().drop(columns_dropped, axis=1)

print(f"Columns before: {df_trimmed.shape[1]}")
print(f"Columns after: {d2.shape[1]}")
print(f"Remaining columns: {list(d2.columns)}")

In [ ]:
# checking for null values
d2.info()

## Null Value Handling

In [ ]:
# populating null values instead of completely dropping using forward fill (time-series) with a 3 hour gap for similar results (can't use KNN because data too large)

import numpy as np

# sort first
d2 = d2.sort_values(['City', 'Start_Time'])

# change to datetime (valuerror showed to add in format='ISO8601' as a parameter)
d2['Start_Time'] = pd.to_datetime(d2['Start_Time'], format='ISO8601')
d2['End_Time'] = pd.to_datetime(d2['End_Time'], format='ISO8601')

# select weather columns
weather_cols = ['Temperature(F)', 'Humidity(%)', 'Pressure(in)', 
                'Visibility(mi)', 'Wind_Direction', 
                'Wind_Speed(mph)', 'Weather_Condition']

# tweak this threshold as needed, currently 3 hours
max_gap = pd.Timedelta(hours=3)

# forward fill weather values, but only if the time gap between the current row and the last known value is within the max_gap threshold
def ffill_with_time_limit(group, cols, max_gap):
    for col in cols:
        filled = group[col].ffill()

        # calculate time diff between each row and the last non-null row
        last_valid_time = group['Start_Time'].where(group[col].notna()).ffill()
        time_gap = group['Start_Time'] - last_valid_time

        # only accept the fill if the gap is within threshold
        group[col] = filled.where(time_gap <= max_gap, other=np.nan)
    return group

# apply the function to each city group (done by city to avoid filling across different cities which may have different weather conditions)
city_groups = []
for city, group in d2.groupby('City'):
    filled_group = ffill_with_time_limit(group, weather_cols, max_gap)
    city_groups.append(filled_group)

# concatenate the groups back together
d2 = pd.concat(city_groups).sort_values(['City', 'Start_Time'])

# precipitation still just fills with 0
d2['Precipitation(in)'] = d2['Precipitation(in)'].fillna(0)

# check remaining nulls
print(d2.isnull().sum())

In [ ]:
# drop the remaining null columns
d2 = d2.dropna()
# reset the index for simplicity
d2.reset_index(drop=True)

## Feature Engineering

In [ ]:
# Extracting more relevant predictors from the start time, such as hour, day of week, weekend, etc
d2['Start_Date'] = d2['Start_Time'].dt.date
d2['Hour'] = d2['Start_Time'].dt.hour
d2["Day_of_Week"] = d2['Start_Time'].dt.dayofweek
d2['Month'] = d2['Start_Time'].dt.month
d2['Weekend'] = d2['Start_Time'].dt.weekday >= 5

In [ ]:
import holidays

# create a US holidays object
us_holidays = holidays.UnitedStates()
# create a Holiday column that indicates whether the date is a holiday
d2['Holiday'] = d2['Start_Date'].apply(lambda x: x in us_holidays).astype(int)
d2['Holiday_Name'] = d2['Start_Time'].dt.date.apply(lambda x: us_holidays.get(x))
d2["Holiday_Name"].fillna("Not Holiday", inplace=True)

In [ ]:
d2[d2['Holiday'] == 1]

### Converting Data to Numerical

In [ ]:
#pd.set_option('display.max_columns', None)

In [ ]:
# One Hot Encoding the cities column
cities = pd.get_dummies(d2['City'], prefix='City')
d2 = pd.concat([d2, cities], axis=1)
d2

In [ ]:
# Since there is too many weather conditions, must consolidate them before doing one hot encoding

weather_conditions_map = {
    'Fair': 'Clear',
    'Clear': 'Clear',
    'Fair / Windy': 'Clear',

    'Mostly Cloudy': 'Cloudy',
    'Partly Cloudy': 'Cloudy',
    'Cloudy': 'Cloudy',
    'Overcast': 'Cloudy',
    'Scattered Clouds': 'Cloudy',
    'Mostly Cloudy / Windy': 'Cloudy',
    'Cloudy / Windy': 'Cloudy',
    'Partly Cloudy / Windy': 'Cloudy',

    'Light Rain': 'Rain/Drizzle',
    'Rain': 'Rain/Drizzle',
    'Heavy Rain': 'Rain/Drizzle',
    'Light Rain / Windy': 'Rain/Drizzle',
    'Light Drizzle': 'Rain/Drizzle',
    'Drizzle': 'Rain/Drizzle',
    'Rain / Windy': 'Rain/Drizzle',
    'Heavy Rain / Windy': 'Rain/Drizzle',
    'Drizzle and Fog': 'Rain/Drizzle',
    'Showers in the Vicinity': 'Rain/Drizzle',
    'Light Rain Showers': 'Rain/Drizzle',
    'Rain Showers': 'Rain/Drizzle',

    'Thunder in the Vicinity': 'Stormy',
    'Thunder': 'Stormy',
    'T-Storm': 'Stormy',
    'Heavy T-Storm': 'Stormy',
    'Thunderstorm': 'Stormy',
    'Light Rain with Thunder': 'Stormy',
    'Light Thunderstorms and Rain': 'Stormy',
    'Heavy Thunderstorms and Rain': 'Stormy',
    'Thunderstorms and Rain': 'Stormy',
    'Heavy T-Storm / Windy': 'Stormy',
    'T-Storm / Windy': 'Stormy',
    'Thunder / Windy': 'Stormy',
    'Squalls / Windy': 'Stormy',
    'Funnel Cloud': 'Stormy',

    'Haze': 'Visibility Issues',
    'Fog': 'Visibility Issues',
    'Shallow Fog': 'Visibility Issues',
    'Mist': 'Visibility Issues',
    'Patches of Fog': 'Visibility Issues',
    'Smoke': 'Visibility Issues',
    'Haze / Windy': 'Visibility Issues',

    'Light Snow': 'Snow/Ice',
    'Snow': 'Snow/Ice',
    'Light Snow and Sleet / Windy': 'Snow/Ice',
    'Wintry Mix': 'Snow/Ice',
    'Wintry Mix / Windy': 'Snow/Ice',
    'Light Sleet': 'Snow/Ice',
    'Heavy Sleet': 'Snow/Ice',
    'Snow and Sleet / Windy': 'Snow/Ice',
    'Heavy Snow': 'Snow/Ice',
    'Light Ice Pellets': 'Snow/Ice',
    'Light Freezing Rain': 'Snow/Ice',
    'Freezing Rain / Windy': 'Snow/Ice',

    'Widespread Dust': 'Dust/Windy',
    'Blowing Dust': 'Dust/Windy',
    'Blowing Dust / Windy': 'Dust/Windy',
    'Duststorm': 'Dust/Windy',
    
}

d2["Weather_Condition_Grouped"] = d2["Weather_Condition"].map(weather_conditions_map).fillna("Other")

In [ ]:
conditions = pd.get_dummies(d2["Weather_Condition_Grouped"], prefix="Weather")
d2 = pd.concat([d2, conditions], axis=1)

In [ ]:
# Converting Sunrise_Sunset to a binary variable for day and night
day_map = {
    'Night': 0,
    'Day': 1
}

d2['Day/Night'] = d2['Sunrise_Sunset'].map(day_map)

### Zone ID Creation

In [ ]:
import h3

# Uses Uber's geospatial indexing library to convert lat/lng to hexagonal zones for better grouping
def get_hex_id(row):
    lat = row['Start_Lat']
    lng = row['Start_Lng']
    return h3.latlng_to_cell(lat, lng, res=6)
    

# Create the Zone_ID to plot the longitude and latitude into hexagonal zones
d2['Zone_ID'] = d2.apply(get_hex_id, axis=1)

# Making the zone into a simple integer
d2['Zone_Int_ID'], _ = pd.factorize(d2['Zone_ID'])

In [ ]:
import plotly.express as px
import h3

# Aggregate accidents by the different zones in Houston - change to different cities for diff results
houston_hex = d2[d2['City_Houston'] == 1].groupby('Zone_ID').size().reset_index(name='Accident_Count')

# Using GeoJSON to plot the hexagonal zones on the map with Plotly
features = []
for zone in houston_hex['Zone_ID']:
    boundary = h3.cell_to_boundary(zone)
    coords = [[lon, lat] for lat, lon in boundary]
    coords.append(coords[0])
    features.append({
        'type': 'Feature',
        'id': zone,
        'properties': {},
        'geometry': {'type': 'Polygon', 'coordinates': [coords]}
    })

geojson = {'type': 'FeatureCollection', 'features': features}

# Plotting the hexagonal zones with accident counts 
fig = px.choropleth_map(
    houston_hex,
    geojson=geojson,
    locations='Zone_ID',
    featureidkey='id',
    color='Accident_Count',
    color_continuous_scale='Reds',
    map_style='carto-positron',
    zoom=10,
    center={'lat': 29.7604, 'lon': -95.3698}, # Coordinates for Houston
    #center={'lat': 25.7617, 'lon': -80.1918}, # Coordinates for Miami
    #center={'lat': 34.0522, 'lon': -118.2437}, # Coordinates for Los Angeles
    labels={'Accident_Count': 'Accidents'}
)

fig.update_layout()
fig.show()

## Creating the Aggregated Dataset

In [ ]:
# Year col for grouping by year
d2['Year'] = d2['Start_Time'].dt.year

poi_cols = ['Amenity', 'Crossing', 'Give_Way', 'Junction', 'Railway', 'Station', 'Stop', 'Traffic_Signal']

# Using mean to create a probability of above events occuring in a given year and zone
zone_year_profile = d2.groupby(['Zone_Int_ID', 'Year'])[poi_cols].mean().reset_index()
city_flags = d2.groupby('Zone_Int_ID')[['City_Houston', 'City_Miami']].max().reset_index() 
zone_year_profile = zone_year_profile.merge(city_flags, on='Zone_Int_ID', how='left')

# Making sure that each zone has a record for each year, even if there were no accidents in that year 
years = sorted(d2['Year'].unique())
zones = zone_year_profile['Zone_Int_ID'].unique()
all_combinations = pd.MultiIndex.from_product([zones, years], names=['Zone_Int_ID', 'Year']).to_frame(index=False) 

# Merge our sparse data into this full skeleton
full_zone_profile = all_combinations.merge(zone_year_profile, on=['Zone_Int_ID', 'Year'], how='left')

# Forward filling and backward filling the missing values for these columns
full_zone_profile = full_zone_profile.sort_values(['Zone_Int_ID', 'Year'])
full_zone_profile[poi_cols] = full_zone_profile.groupby('Zone_Int_ID')[poi_cols].ffill().bfill()
full_zone_profile[['City_Houston', 'City_Miami']] = full_zone_profile.groupby('Zone_Int_ID')[['City_Houston', 'City_Miami']].ffill().bfill()

full_zone_profile

In [ ]:
# Making 2h interval time slowts
d2['Time_Stamp'] = d2['Start_Time'].dt.floor('2H')
min_time = d2['Time_Stamp'].min()
max_time = d2['Time_Stamp'].max()
all_times = pd.date_range(start=min_time, end=max_time, freq='2h')

# Create a DataFrame with all the time slots, so that we can merge after and still have times with no accidents
time_df = pd.DataFrame({'Time_Stamp': all_times})

In [ ]:
time_df['Year'] = time_df['Time_Stamp'].dt.year
time_df['Hour'] = time_df['Time_Stamp'].dt.hour
time_df['Day_of_Week'] = time_df['Time_Stamp'].dt.dayofweek
time_df['Month'] = time_df['Time_Stamp'].dt.month
time_df['Weekend'] = (time_df['Day_of_Week'] >= 5).astype(int)

time_df['Holiday'] = time_df['Time_Stamp'].dt.date.apply(lambda x: int(x in us_holidays))

In [ ]:
# Merging the time data with the zone profile data to create a value for each row
master_grid = time_df.merge(full_zone_profile, on='Year', how='inner')

In [ ]:
weather_metrics = ['Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)']
weather_ohe = ['Weather_Clear', 'Weather_Cloudy', 'Weather_Dust/Windy', 'Weather_Rain/Drizzle', 'Weather_Snow/Ice', 'Weather_Stormy', 'Weather_Visibility Issues']

all_weather = weather_metrics + weather_ohe

# Getting the mean values for all weather cols across the whole city - gives real value for metrics and pribability for the one hot encoded conditions 
city_weather_ref = d2.groupby(['City', 'Time_Stamp'])[all_weather].mean().reset_index()

master_grid['City'] = master_grid[['City_Houston', 'City_Miami']].idxmax(axis=1).str.replace('City_', '')

master_grid = (master_grid.merge(city_weather_ref, on=['City', 'Time_Stamp'], how='left').sort_values(['City', 'Time_Stamp']))

# 4. Fill gaps and downcast to float32 immediately to save RAM
master_grid[all_weather] = (master_grid.groupby('City')[all_weather] .ffill().bfill().astype(float))



In [ ]:
# Finding the number of accidents in each zone and time slot to create the target variable for our model
accident_counts = d2.groupby(['Zone_Int_ID', 'Time_Stamp']).size().reset_index(name='Accident_Count')

master_grid = master_grid.merge(accident_counts, on=['Zone_Int_ID', 'Time_Stamp'], how='left')

# If no accident was found in that 2-hour window, the count is zero
master_grid['Accident_Count'] = master_grid['Accident_Count'].fillna(0).astype(int)


In [ ]:
modelling_dataset = master_grid.drop(columns=["City"]).copy()

modelling_dataset

In [ ]:
# # 2 hour interval buckets
# d2['Time_Stamp'] = d2['Start_Time'].dt.floor('2h')
# # Extract hour directly from Time_Stamp so it matches the 2h bucket
# d2['Hour'] = d2['Time_Stamp'].dt.hour

# # Aggregate accidents into zone x 2hr time slot buckets (and only keeping necessary coloumns for modelling)
# # mean: weather columns (avg across accidents in bucket)
# # max: POI/weather flags (1 if any accident in bucket had it)
# # first: categorical columns (same value for all accidents in bucket)
# modelling_df = d2.groupby(['Zone_Int_ID', 'Time_Stamp']).agg(**{
#     'Mean_Severity'        : ('Severity', 'mean'),
#     'Mean_Distance(mi)'    : ('Distance(mi)', 'mean'),
#     'Mean_Temp(F)'         : ('Temperature(F)', 'mean'),
#     'Mean_Humidity(%)'     : ('Humidity(%)', 'mean'),
#     'Mean_Pressure(in)'    : ('Pressure(in)', 'mean'),
#     'Mean_Visibility(mi)'  : ('Visibility(mi)', 'mean'),
#     'Mean_Wind(mph)'       : ('Wind_Speed(mph)', 'mean'),
#     'Mean_Precip(in)'      : ('Precipitation(in)', 'mean'),
#     'Amenity'              : ('Amenity', 'max'),
#     'Crossing'             : ('Crossing', 'max'),
#     'Give_Way'             : ('Give_Way', 'max'),
#     'Junction'             : ('Junction', 'max'),
#     'Railway'              : ('Railway', 'max'),
#     'Station'              : ('Station', 'max'),
#     'Stop'                 : ('Stop', 'max'),
#     'Traffic_Signal'       : ('Traffic_Signal', 'max'),
#     'Hour'                 : ('Hour', 'first'),
#     'Day_of_Week'          : ('Day_of_Week', 'first'),
#     'Month'                : ('Month', 'first'),
#     'Weekend'              : ('Weekend', 'first'),
#     'Holiday'              : ('Holiday', 'first'),
#     'City_Houston'         : ('City_Houston', 'first'),
#     'City_LA'              : ('City_Los Angeles', 'first'),
#     'City_Miami'           : ('City_Miami', 'first'),
#     'Weather_Clear'        : ('Weather_Clear', 'max'),
#     'Weather_Cloudy'       : ('Weather_Cloudy', 'max'),
#     'Weather_Dust/Windy'   : ('Weather_Dust/Windy', 'max'),
#     'Weather_Rain/Drizzle' : ('Weather_Rain/Drizzle', 'max'),
#     'Weather_Snow/Ice'     : ('Weather_Snow/Ice', 'max'),
#     'Weather_Stormy'       : ('Weather_Stormy', 'max'),
#     'Weather_Visibility'   : ('Weather_Visibility Issues', 'max'),
#     'Day_Night'            : ('Day/Night', 'first'),
# }).reset_index()

# # Add Accident_Count target variable
# accident_counts = d2.groupby(['Zone_Int_ID', 'Time_Stamp']).size().reset_index(name='Accident_Count')
# modelling_df = modelling_df.merge(accident_counts, on=['Zone_Int_ID', 'Time_Stamp'], how='left')

In [ ]:
# modelling_df.head()

In [ ]:
# modelling_dataset.to_csv("modelling_dataset.csv", index=False)

In [ ]:
modelling_dataset.to_parquet('modelling_dataset.parquet', engine='fastparquet')